In [ ]:
# One-click Git push from Jupyter
# Works with your existing Git credentials (SSH or stored HTTPS).
# Edit REPO and (optionally) COMMIT_MESSAGE below.

from pathlib import Path
from datetime import datetime
import subprocess, shlex, sys

# === SETTINGS ===
REPO = Path("/Users/ousmane/Desktop/2025 - Documentations/Python/Economics/genesis-research")
COMMIT_MESSAGE = f"push ({datetime.now().strftime('%Y-%m-%d %H:%M')})"
# =================

def run(cmd, cwd):
    """Run a shell command, stream output, and raise on error."""
    p = subprocess.Popen(
        shlex.split(cmd),
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    out_lines = []
    for line in p.stdout:
        print(line, end="")
        out_lines.append(line)
    ret = p.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed ({ret}): {cmd}\n{''.join(out_lines)}")
    return "".join(out_lines)

def current_branch(cwd):
    try:
        return run("git rev-parse --abbrev-ref HEAD", cwd).strip()
    except Exception:
        return None

def has_upstream(cwd):
    try:
        run("git rev-parse --abbrev-ref --symbolic-full-name @{u}", cwd)
        return True
    except Exception:
        return False

def repo_ok(cwd):
    try:
        run("git rev-parse --is-inside-work-tree", cwd)
        return True
    except Exception:
        return False

def git_push(repo_path=REPO, message=COMMIT_MESSAGE):
    repo_path = Path(repo_path)
    if not repo_ok(repo_path):
        print(f"✗ Not a git repo: {repo_path}")
        return

    print(f"➤ Repository: {repo_path}")
    run("git status -sb", repo_path)

    # Stage changes
    run("git add -A", repo_path)

    # Commit only if there are staged changes
    status = run("git status --porcelain", repo_path)
    if not status.strip():
        print("✓ No changes to commit.")
    else:
        try:
            run(f'git commit -m "{message}"', repo_path)
        except RuntimeError as e:
            # If nothing to commit (e.g., hooks), surface message but keep going
            if "nothing to commit" in str(e).lower():
                print("✓ Nothing to commit (clean).")
            else:
                raise

    # Push
    br = current_branch(repo_path) or "main"
    if has_upstream(repo_path):
        run("git push", repo_path)
    else:
        # First push for this branch — set upstream
        print(f"ℹ️ No upstream set. Pushing with -u to origin {br}…")
        run(f"git push -u origin {br}", repo_path)

    print("✅ Done.")

# --- OPTIONAL: a nice button in Jupyter ---
try:
    from ipywidgets import Button, Output, VBox
    from IPython.display import display

    btn = Button(description="Push to Git", tooltip="git add/commit/push", button_style="")
    out = Output()

    def on_click(_):
        out.clear_output()
        with out:
            try:
                git_push()
            except Exception as e:
                print("❌ Error:", e, file=sys.stderr)

    btn.on_click(on_click)
    display(VBox([btn, out]))
except Exception:
    # If ipywidgets isn't available, just call git_push() directly:
    pass

# Uncomment to push immediately when you run the cell (no button click needed):
# git_push()